# Fine-tuning CodeBERT for the risk-controlled triage study
Reproduces the study's 7 scenarios on the 30 000-function subsample with a **fine-tuned** CodeBERT and writes per-function scores (`ft_cb`) that plug into the existing analysis (`src/merge_ft.py`).

**Before you start:** *Runtime → Change runtime type → GPU* (T4 is enough; A100/L4 is faster). Keep the notebook tab open; the run is resumable, so after a disconnect just re-run the cells from "Run".

Rough cost (estimate, not measured): 10–20 min per scenario-seed on a T4 → about 1.5–2.5 h for seed 0 (7 runs), about 5–7 h for all three seeds.

In [ ]:
!nvidia-smi -L

## 1. Configuration

In [ ]:
#@title Configuration
REPO_URL = "https://github.com/deep1789/Automated-SE.git"   #@param {type:"string"}
BRANCH   = "claude/optimistic-maxwell-qfng5l"                #@param {type:"string"}
GITHUB_TOKEN = ""   #@param {type:"string"}   # only needed if the repo is private (use a fine-grained read-only token, never commit it)
USE_DRIVE = True    #@param {type:"boolean"}   # strongly recommended: results survive disconnects
OUT = "/content/drive/MyDrive/ft_codebert" if USE_DRIVE else "/content/ft_codebert"

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
import os; os.makedirs(OUT, exist_ok=True); print("results will be written to", OUT)

## 2. Get the code
Either clone the repository, or (fallback) upload `colab/bundle.zip` from the repository.

In [ ]:
import os, subprocess
if not os.path.exists("/content/Automated-SE"):
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    r = subprocess.run(["git", "clone", "-q", "--branch", BRANCH, url, "/content/Automated-SE"], capture_output=True, text=True)
    if r.returncode != 0:
        print("git clone failed (private repo?). Upload colab/bundle.zip instead:\n", r.stderr[-300:].replace(GITHUB_TOKEN or "###", "***"))
        from google.colab import files; up = files.upload()
        os.makedirs("/content/Automated-SE", exist_ok=True)
        import zipfile; zipfile.ZipFile(list(up)[0]).extractall("/content/Automated-SE")
%cd /content/Automated-SE
!ls

In [ ]:
!pip -q install huggingface_hub pyarrow scikit-learn lightgbm transformers

## 3. Prepare data (about 5 min)
Downloads DiverseVul and Big-Vul from the Hugging Face hub, applies the study's normalisation/deduplication, rebuilds the 30 000-function subsample and **verifies it against the study's checksum** (it stops if the data differ).

In [ ]:
!python colab/finetune_codebert.py --prepare

## 4. Dry run (1 minute)
Checks splits, I/O and resume logic without a model.

In [ ]:
!python colab/finetune_codebert.py --dummy --scenarios DV-P BV2DV --seeds 0 --out /content/dummy_out 2>&1 | grep -v -i warn

## 5. Run: seed 0, all seven scenarios
Resumable: finished scenario-seed pairs are skipped. Add `--max-len 512` on an A100/L4 for longer context (changes the experiment; report it).

In [ ]:
!python colab/finetune_codebert.py --seeds 0 --out $OUT 2>&1 | grep -v -i "warn\|unauth\|UNEXPECTED\|MISSING\|Notes\|checkpoint\|^$" 

## 6. (Optional) seeds 1 and 2
The paper's other experiments use three seeds; running these takes several more hours.

In [ ]:
!python colab/finetune_codebert.py --seeds 1 2 --out $OUT 2>&1 | grep -v -i "warn\|unauth\|UNEXPECTED\|MISSING\|Notes\|checkpoint\|^$" 

## 7. Quick look and download

In [ ]:
import glob, json, pandas as pd
rows = [json.load(open(f)) for f in sorted(glob.glob(f"{OUT}/*_done.json"))]
print(pd.DataFrame([dict(scenario=r["scenario"], seed=r["seed"], best_val_auroc=round(r.get("best_val_auroc", float("nan")), 3), minutes=round(r["seconds"]/60, 1)) for r in rows]))
!cd $OUT && zip -q -r /content/ft_codebert_scores.zip . && ls -la /content/ft_codebert_scores.zip
from google.colab import files; files.download("/content/ft_codebert_scores.zip")

## 8. Back in the repository
```
unzip ft_codebert_scores.zip -d ft_out
python src/merge_ft.py ft_out            # aligns on function hash and adds the ft_cb column to results/cbn_cache
python src/analyze_cb.py norm            # detection + triage tables incl. the fine-tuned model
python src/make_cb.py                    # tables and figure pick up the fine-tuned column automatically
```